# Module 13: Structured Streaming

Structured Streaming lets you process **real-time data** using the same DataFrame API you already know. The key idea: treat a live data stream as an **unbounded table** that keeps growing.

```
Batch:     Read entire file  →  Transform  →  Write result
Streaming: Read new data     →  Transform  →  Update result (continuously)
```

In this module you'll learn:
1. **Streaming concepts** — unbounded tables, triggers, output modes
2. **Reading a stream** — `readStream` from files
3. **Transformations** — same DataFrame API
4. **Output modes** — append, complete, update
5. **Writing a stream** — `writeStream` to memory/console sinks
6. **Windowed aggregations** — tumbling and sliding time windows

**Note**: We'll simulate streaming by writing files into a directory. In production, you'd read from Kafka, Kinesis, or other message queues.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum, avg, count, round, window,
    current_timestamp, to_timestamp, expr
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType, TimestampType
)
import os
import shutil
import time

spark = SparkSession.builder \
    .appName("Module 13 - Structured Streaming") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")

---
## Concept 1: The Streaming Mental Model

In Structured Streaming, every new piece of data is treated as a **new row appended** to an unbounded input table.

```
Time 0:  Input Table = [row1, row2, row3]
Time 1:  Input Table = [row1, row2, row3, row4, row5]        ← 2 new rows
Time 2:  Input Table = [row1, row2, row3, row4, row5, row6]  ← 1 new row
```

Your query runs on this table — Spark figures out how to incrementally process only the new data.

**Three output modes**:

| Mode | What's written | Use when |
|------|---------------|----------|
| `append` | Only new rows | No aggregations, or windowed aggregations |
| `complete` | Entire result table | Aggregations (groupBy) — rewrites everything each time |
| `update` | Only changed rows | Aggregations where you only want the delta |

**Interview tip**: Know the difference between the three modes and when each is appropriate.

---
## Concept 2: Simulating a Stream from Files

In production, streams come from Kafka, Kinesis, etc. For learning, we'll simulate a stream by **dropping CSV files into a directory**. Spark's `readStream` will pick up each new file as it appears.

We must define a **schema** upfront — streaming sources can't infer schema (there might be no data yet).

In [ ]:
# Create directories for our simulated stream
stream_input = "../output/stream_input"
stream_checkpoint = "../output/stream_checkpoint"

# Clean up from previous runs
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print(f"Input directory: {stream_input}")
print(f"Checkpoint directory: {stream_checkpoint}")

In [ ]:
# Define schema for our streaming sales data
sales_schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("sale_time", StringType(), True)
])

# Create a streaming DataFrame — reads from the directory
stream_df = spark.readStream \
    .schema(sales_schema) \
    .option("header", "true") \
    .csv(stream_input)

print(f"Is streaming: {stream_df.isStreaming}")
stream_df.printSchema()

---
## Concept 3: Transformations on Streams

The best part: you use the **exact same DataFrame API** for streaming as for batch. `.filter()`, `.select()`, `.groupBy()`, `.join()` — all work.

The only difference is how you **read** (readStream) and **write** (writeStream).

In [ ]:
# Apply transformations — same syntax as batch!
high_value_sales = stream_df \
    .filter(col("amount") > 1000) \
    .select("product", "amount", "region")

# Aggregation on the stream
region_totals = stream_df \
    .groupBy("region") \
    .agg(
        round(sum("amount"), 2).alias("total_amount"),
        count("*").alias("num_sales")
    )

print("Transformations defined (not running yet — lazy evaluation applies to streams too!)")

---
## Concept 4: Writing a Stream — Memory Sink

To start processing, you call `writeStream`. The **memory sink** writes results to an in-memory table you can query with SQL — perfect for development and testing.

**Key parameters**:
- `outputMode` — append, complete, or update
- `format` — where to write (memory, console, parquet, kafka, etc.)
- `queryName` — name for the in-memory table
- `checkpointLocation` — where Spark tracks progress (required for fault tolerance)

In [ ]:
# Start the streaming query — writes aggregation results to memory
query = region_totals.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("region_sales") \
    .option("checkpointLocation", stream_checkpoint + "/region_sales") \
    .start()

print(f"Query running: {query.isActive}")
print(f"Query name: {query.name}")

In [ ]:
# No data yet — table is empty
spark.sql("SELECT * FROM region_sales").show()

In [ ]:
# Now let's simulate incoming data by writing CSV files to the input directory
# Batch 1: First set of sales
batch1 = spark.createDataFrame([
    (1, "Widget A", 1250.0, 5, "West", "2024-01-15 10:00:00"),
    (2, "Gadget X", 2100.0, 1, "East", "2024-01-15 10:05:00"),
    (3, "Widget B", 890.5, 3, "West", "2024-01-15 10:10:00"),
    (4, "Widget A", 625.0, 2, "Central", "2024-01-15 10:15:00"),
], sales_schema)

batch1.write.mode("append").option("header", "true").csv(stream_input)
print("Batch 1 written!")

# Give Spark a moment to pick up the new file
time.sleep(3)

# Check results
spark.sql("SELECT * FROM region_sales ORDER BY region").show()

In [ ]:
# Batch 2: More sales arrive
batch2 = spark.createDataFrame([
    (5, "Gadget Y", 1780.0, 2, "East", "2024-01-15 11:00:00"),
    (6, "Widget C", 150.0, 10, "West", "2024-01-15 11:05:00"),
    (7, "Widget A", 3750.0, 15, "Central", "2024-01-15 11:10:00"),
], sales_schema)

batch2.write.mode("append").option("header", "true").csv(stream_input)
print("Batch 2 written!")

time.sleep(3)

# The aggregations are UPDATED with the new data
spark.sql("SELECT * FROM region_sales ORDER BY region").show()

In [ ]:
# Stop the query when done
query.stop()
print("Query stopped.")

Notice how the totals **grew** as new data arrived — that's the streaming magic. The `complete` mode rewrites the entire result table each time.

#### Try It: Stream Product Totals

1. Clean the input and checkpoint directories (code provided below)
2. Create a streaming DataFrame from the input directory using `sales_schema`
3. Group by `product` and aggregate `sum(amount)` and `count(*)`
4. Write to memory sink with `queryName="product_sales"` and `complete` mode
5. Write 2 batches of data and check results after each

In [ ]:
# Clean up for the exercise
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Your code here


In [ ]:
# --- Solution ---
# Clean up
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Read stream
product_stream = spark.readStream \
    .schema(sales_schema) \
    .option("header", "true") \
    .csv(stream_input)

# Aggregate by product
product_totals = product_stream \
    .groupBy("product") \
    .agg(
        round(sum("amount"), 2).alias("total_amount"),
        count("*").alias("num_sales")
    )

# Start query
product_query = product_totals.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("product_sales") \
    .option("checkpointLocation", stream_checkpoint + "/product_sales") \
    .start()

# Batch 1
spark.createDataFrame([
    (1, "Widget A", 1250.0, 5, "West", "2024-01-15 10:00:00"),
    (2, "Gadget X", 2100.0, 1, "East", "2024-01-15 10:05:00"),
    (3, "Widget A", 625.0, 2, "Central", "2024-01-15 10:10:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)

time.sleep(3)
print("After batch 1:")
spark.sql("SELECT * FROM product_sales ORDER BY product").show()

# Batch 2
spark.createDataFrame([
    (4, "Widget A", 750.0, 3, "West", "2024-01-15 11:00:00"),
    (5, "Gadget X", 4200.0, 2, "East", "2024-01-15 11:05:00"),
    (6, "Widget B", 890.0, 3, "Central", "2024-01-15 11:10:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)

time.sleep(3)
print("After batch 2:")
spark.sql("SELECT * FROM product_sales ORDER BY product").show()

product_query.stop()

---
## Concept 5: Output Modes in Practice

Let's compare `complete` vs `update` vs `append` to see the difference.

In [ ]:
# Clean up
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Set up two identical streaming queries with different output modes
stream_a = spark.readStream.schema(sales_schema).option("header", "true").csv(stream_input)
stream_b = spark.readStream.schema(sales_schema).option("header", "true").csv(stream_input)

agg_a = stream_a.groupBy("region").agg(count("*").alias("cnt"))
agg_b = stream_b.groupBy("region").agg(count("*").alias("cnt"))

# Complete mode — entire result each time
q_complete = agg_a.writeStream \
    .outputMode("complete").format("memory") \
    .queryName("mode_complete") \
    .option("checkpointLocation", stream_checkpoint + "/complete").start()

# Update mode — only changed rows
q_update = agg_b.writeStream \
    .outputMode("update").format("memory") \
    .queryName("mode_update") \
    .option("checkpointLocation", stream_checkpoint + "/update").start()

print("Both queries running.")

In [ ]:
# Send data and compare
spark.createDataFrame([
    (1, "Widget A", 100.0, 1, "West", "2024-01-15 10:00:00"),
    (2, "Widget B", 200.0, 2, "East", "2024-01-15 10:05:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)

time.sleep(3)

print("COMPLETE mode (full result):")
spark.sql("SELECT * FROM mode_complete ORDER BY region").show()

print("UPDATE mode (only changed rows — same result here since all rows are new):")
spark.sql("SELECT * FROM mode_update ORDER BY region").show()

In [ ]:
# Send more data — only West region
spark.createDataFrame([
    (3, "Widget C", 300.0, 3, "West", "2024-01-15 11:00:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)

time.sleep(3)

print("COMPLETE mode (still shows ALL regions):")
spark.sql("SELECT * FROM mode_complete ORDER BY region").show()

print("UPDATE mode (in memory sink, shows accumulated state):")
spark.sql("SELECT * FROM mode_update ORDER BY region").show()

# Note: the difference is more visible with console or file sinks.
# In memory sink, update mode accumulates state for SQL querying.
# With a file sink, update mode would only write the West row this batch.

q_complete.stop()
q_update.stop()

---
## Concept 6: Windowed Aggregations on Event Time

In streaming, you often want to aggregate over **time windows**: "total sales per 10-minute window" or "average amount per hour".

Spark provides the `window()` function for this:

- **Tumbling window** — fixed, non-overlapping intervals (e.g., every 1 hour)
- **Sliding window** — overlapping intervals (e.g., 1 hour window, sliding every 30 min)

```
Tumbling (1 hour):  |---1---|---2---|---3---|---4---|
Sliding (1hr/30m):  |---1---|---2---|---3---|---4---|
                        |---1.5-|---2.5-|---3.5-|
```

In [ ]:
# Clean up
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Read stream and parse the timestamp
time_stream = spark.readStream \
    .schema(sales_schema) \
    .option("header", "true") \
    .csv(stream_input) \
    .withColumn("event_time", to_timestamp("sale_time"))

# Tumbling window: 1-hour windows on event_time
windowed_agg = time_stream \
    .groupBy(
        window(col("event_time"), "1 hour"),  # tumbling window
        col("region")
    ) \
    .agg(
        round(sum("amount"), 2).alias("total_amount"),
        count("*").alias("num_sales")
    )

# Start query
window_query = windowed_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("windowed_sales") \
    .option("checkpointLocation", stream_checkpoint + "/windowed") \
    .start()

print("Windowed query running.")

In [ ]:
# Sales across different hours
spark.createDataFrame([
    (1, "Widget A", 1250.0, 5, "West",    "2024-01-15 09:15:00"),
    (2, "Gadget X", 2100.0, 1, "East",    "2024-01-15 09:45:00"),
    (3, "Widget B", 890.5,  3, "West",    "2024-01-15 10:10:00"),
    (4, "Widget A", 625.0,  2, "East",    "2024-01-15 10:30:00"),
    (5, "Gadget Y", 1780.0, 2, "West",    "2024-01-15 10:55:00"),
    (6, "Widget C", 3200.0, 8, "Central", "2024-01-15 11:05:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)

time.sleep(3)

# Results grouped by 1-hour windows AND region
spark.sql("""
    SELECT window.start, window.end, region, total_amount, num_sales
    FROM windowed_sales
    ORDER BY window.start, region
""").show(truncate=False)

In [ ]:
window_query.stop()

Each row shows the time window (`start` to `end`), the region, and the aggregate. Events are grouped into the correct 1-hour bucket based on their `event_time`.

For a **sliding window**, pass a second duration:
```python
window(col("event_time"), "1 hour", "30 minutes")  # 1-hour window, slides every 30 min
```

#### Try It: Sliding Window Aggregation

1. Clean the input/checkpoint directories
2. Create a streaming query with a **30-minute sliding window** that slides every **15 minutes**
3. Group by the window and compute `sum(amount)` and `count(*)`
4. Write to memory sink, send some test data, and check results

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
# Clean up
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

sliding_stream = spark.readStream \
    .schema(sales_schema) \
    .option("header", "true") \
    .csv(stream_input) \
    .withColumn("event_time", to_timestamp("sale_time"))

# Sliding window: 30 min window, slides every 15 min
sliding_agg = sliding_stream \
    .groupBy(
        window(col("event_time"), "30 minutes", "15 minutes")
    ) \
    .agg(
        round(sum("amount"), 2).alias("total_amount"),
        count("*").alias("num_sales")
    )

sliding_query = sliding_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("sliding_sales") \
    .option("checkpointLocation", stream_checkpoint + "/sliding") \
    .start()

# Send test data
spark.createDataFrame([
    (1, "Widget A", 500.0,  2, "West", "2024-01-15 10:05:00"),
    (2, "Widget B", 800.0,  3, "East", "2024-01-15 10:20:00"),
    (3, "Gadget X", 1200.0, 1, "West", "2024-01-15 10:35:00"),
    (4, "Widget A", 300.0,  1, "East", "2024-01-15 10:50:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)

time.sleep(3)

# Each event may appear in multiple overlapping windows
spark.sql("""
    SELECT window.start, window.end, total_amount, num_sales
    FROM sliding_sales
    ORDER BY window.start
""").show(truncate=False)

sliding_query.stop()

Notice how a single event can appear in **multiple overlapping windows** with sliding windows. That's the key difference from tumbling windows.

---
## Capstone Exercise

Build a streaming dashboard that tracks **real-time sales by region per hour**:

1. Clean the input/checkpoint directories
2. Read a stream with `sales_schema`, parse `sale_time` to timestamp
3. Create a 1-hour tumbling window grouped by both `window` and `region`
4. Aggregate: `sum(amount)`, `avg(amount)`, `count(*)`
5. Write to memory sink
6. Send 3 batches of data spaced across different hours
7. Query the result after each batch to see the dashboard update

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---

# Clean up
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Read stream
cap_stream = spark.readStream \
    .schema(sales_schema) \
    .option("header", "true") \
    .csv(stream_input) \
    .withColumn("event_time", to_timestamp("sale_time"))

# Windowed aggregation by region
dashboard = cap_stream \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("region")
    ) \
    .agg(
        round(sum("amount"), 2).alias("total"),
        round(avg("amount"), 2).alias("avg_sale"),
        count("*").alias("num_sales")
    )

cap_query = dashboard.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("dashboard") \
    .option("checkpointLocation", stream_checkpoint + "/dashboard") \
    .start()

# Batch 1: Morning sales
spark.createDataFrame([
    (1, "Widget A", 1250.0, 5,  "West",    "2024-01-15 09:10:00"),
    (2, "Gadget X", 2100.0, 1,  "East",    "2024-01-15 09:30:00"),
    (3, "Widget B", 890.0,  3,  "Central", "2024-01-15 09:45:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)
time.sleep(3)
print("=== After Batch 1 (9 AM hour) ===")
spark.sql("SELECT window.start, window.end, region, total, avg_sale, num_sales FROM dashboard ORDER BY window.start, region").show(truncate=False)

# Batch 2: Late morning sales
spark.createDataFrame([
    (4, "Widget A", 625.0,  2,  "West",    "2024-01-15 10:15:00"),
    (5, "Gadget Y", 1780.0, 2,  "East",    "2024-01-15 10:40:00"),
    (6, "Widget C", 450.0,  5,  "West",    "2024-01-15 10:55:00"),
    (7, "Widget A", 3750.0, 15, "Central", "2024-01-15 09:50:00"),  # late-arriving data for 9AM!
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)
time.sleep(3)
print("=== After Batch 2 (10 AM hour + late 9 AM data) ===")
spark.sql("SELECT window.start, window.end, region, total, avg_sale, num_sales FROM dashboard ORDER BY window.start, region").show(truncate=False)

# Batch 3: Afternoon sales
spark.createDataFrame([
    (8,  "Gadget Z", 5500.0, 1, "East",    "2024-01-15 14:00:00"),
    (9,  "Widget B", 1200.0, 4, "Central", "2024-01-15 14:20:00"),
    (10, "Widget A", 900.0,  3, "West",    "2024-01-15 14:45:00"),
], sales_schema).write.mode("append").option("header", "true").csv(stream_input)
time.sleep(3)
print("=== After Batch 3 (2 PM hour) ===")
spark.sql("SELECT window.start, window.end, region, total, avg_sale, num_sales FROM dashboard ORDER BY window.start, region").show(truncate=False)

cap_query.stop()
print("Dashboard stream stopped.")

In [ ]:
# Clean up stream directories
for d in [stream_input, stream_checkpoint]:
    if os.path.exists(d):
        shutil.rmtree(d)

spark.stop()

---
## What You Learned

- **Structured Streaming** treats a data stream as an unbounded table
- **`readStream`** reads from a source; **`writeStream`** writes to a sink
- **Same DataFrame API** — filter, select, groupBy, join all work on streams
- **Output modes**: `append` (new rows only), `complete` (full result), `update` (changed rows)
- **Tumbling windows** — fixed, non-overlapping time intervals
- **Sliding windows** — overlapping intervals for smoother aggregations
- **Checkpoints** — Spark tracks progress for fault tolerance

### Streaming Quick Reference

| Concept | Code |
|---------|------|
| Read stream | `spark.readStream.schema(s).format("csv").load(path)` |
| Write stream | `df.writeStream.outputMode("complete").format("memory").start()` |
| Tumbling window | `window(col("ts"), "1 hour")` |
| Sliding window | `window(col("ts"), "1 hour", "30 minutes")` |
| Stop query | `query.stop()` |
| Check status | `query.isActive`, `query.status` |

### Production Sources & Sinks

| Source/Sink | Format |
|------------|--------|
| Kafka | `format("kafka")` |
| Files (Parquet, CSV, JSON) | `format("parquet")` etc. |
| Console (debug) | `format("console")` |
| Memory (debug) | `format("memory")` |
| Delta Lake | `format("delta")` |